In [2]:
%pip install kaggle -q
import os, json

# Create the folder first
os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_credentials = {
    "username": "naruto17",
    "key": "686f36ec7c4db4a312cd723b82d66a09"
}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d marquis03/cats-and-dogs
!unzip -q cats-and-dogs.zip -d cats_vs_dogs

print("Ready:", os.listdir('cats_vs_dogs'))

Dataset URL: https://www.kaggle.com/datasets/marquis03/cats-and-dogs
License(s): apache-2.0
100% 9.75M/9.75M [00:00<00:00, 91.5MB/s]

Ready: ['val', 'train.csv', 'train', 'val.csv']


In [3]:
import os
import pandas as pd

train_df = pd.read_csv('cats_vs_dogs/train.csv')
val_df = pd.read_csv('cats_vs_dogs/val.csv')

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("\nColumns:", train_df.columns.tolist())
print("\nSample:\n", train_df.head())
print("\nClass distribution:\n", train_df['category'].value_counts())

# Check image folders
print("\nTrain subfolders:", os.listdir('cats_vs_dogs/train'))

Train shape: (275, 2)
Val shape: (70, 2)

Columns: ['image:FILE', 'category']

Sample:
                                           image:FILE  category
0  train/cat/Sphynx_159_jpg.rf.022528b23ac690c34a...         0
1  train/cat/Persian_139_jpg.rf.0e67f7e0a76dc49d0...         0
2  train/cat/Bengal_150_jpg.rf.05c93a40014062c5ae...         0
3  train/cat/Bombay_140_jpg.rf.15757f698af74453f3...         0
4  train/cat/Persian_128_jpg.rf.16da80c477d1ca2bc...         0

Class distribution:
 category
1    180
0     95
Name: count, dtype: int64

Train subfolders: ['dog', 'cat', 'classname.txt']


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [18]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

trainset = datasets.ImageFolder('cats_vs_dogs/train', transform=transform)
valset = datasets.ImageFolder('cats_vs_dogs/val', transform=transform)

In [19]:
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
valloader = DataLoader(valset, batch_size=32, shuffle=False)

In [20]:
class Dogs_vs_Cats_CNN(nn.Module):
    def __init__(self):
        super(Dogs_vs_Cats_CNN, self).__init__()

        self.Conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(16*16*128, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.Conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

    

In [24]:

model = Dogs_vs_Cats_CNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [25]:
epochs = 10

for epoch in range(epochs):

    epoch_loss = 0.0
    model.train()
    for images, labels in trainloader:
        
        labels = labels.float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"epoch={epoch+1} & loss ={epoch_loss/len(trainloader)}")

    

epoch=1 & loss =0.6572923130459256
epoch=2 & loss =0.655939863787757
epoch=3 & loss =0.6317267484135098
epoch=4 & loss =0.6183057957225375
epoch=5 & loss =0.6121529473198785
epoch=6 & loss =0.6098802023463779
epoch=7 & loss =0.5755961868498061
epoch=8 & loss =0.5578332775168948
epoch=9 & loss =0.5498935348457761
epoch=10 & loss =0.5468150443500943


In [27]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in valloader:
        outputs = model(images)
        predicted = (outputs > 0.0).float().squeeze(1) 
        total_labels += labels.size(0)
        correct_labels += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct_labels / total_labels}%")

Accuracy: 65.71428571428571%
